# 🌙 Enhanced Islamic Heritage RAG with Free Models

This notebook enhances your existing RAG system with:
- **Free LLMs** (no API keys needed)
- **Better prompting** for Islamic heritage
- **Source citations** 
- **Interactive interface**

## ⚠️ Disclaimer
This system provides historical and cultural information about Islamic civilization. It does not interpret religious texts or provide theological advice.

In [ ]:
# Install required packages
!pip install -q transformers torch accelerate sentence-transformers chromadb langchain langchain-community wikipedia-api

In [ ]:
# Import libraries
import os
import json
import glob
import warnings
warnings.filterwarnings("ignore")

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from transformers import pipeline
import torch

print("📚 Libraries imported successfully!")
print(f"🔥 GPU available: {torch.cuda.is_available()}")

In [ ]:
# Collect additional Wikipedia data (expand your dataset)
import wikipediaapi

wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='NurAI-Enhanced-RAG/1.0 (educational-use)'
)

# Additional topics to enhance your dataset
new_topics = [
    "Islamic Golden Age",
    "Cordoba Mosque", 
    "Abbasid Caliphate",
    "Al-Andalus",
    "Moorish architecture",
    "Islamic calligraphy",
    "Madrasa",
    "Islamic gardens"
]

os.makedirs("enhanced_data", exist_ok=True)

for topic in new_topics:
    try:
        page = wiki.page(topic)
        if page.exists():
            data = {
                "title": topic,
                "url": page.fullurl,
                "text": page.text
            }
            
            filename = f"enhanced_data/{topic.replace(' ', '_')}.json"
            with open(filename, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
            
            print(f"✅ Added: {topic}")
        else:
            print(f"⚠️ Skipped: {topic}")
    except Exception as e:
        print(f"❌ Error with {topic}: {e}")

print("\n📊 Enhanced dataset ready!")

In [ ]:
# Create enhanced vector database
print("🔧 Creating enhanced vector database...")

# Load all data (original + enhanced)
texts, metadatas = [], []
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,  # Smaller chunks for better precision
    chunk_overlap=100
)

# Load from both data directories
data_dirs = ["data", "enhanced_data"]
for data_dir in data_dirs:
    if os.path.exists(data_dir):
        for path in glob.glob(f"{data_dir}/*.json"):
            with open(path, encoding="utf-8") as f:
                d = json.load(f)
                chunks = splitter.split_text(d["text"])
                for chunk in chunks:
                    texts.append(chunk)
                    metadatas.append({
                        "title": d["title"],
                        "url": d["url"],
                        "source": os.path.basename(path)
                    })

print(f"📝 Processing {len(texts)} text chunks...")

# Create embeddings with multilingual model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Create enhanced vector store
enhanced_db = Chroma.from_texts(
    texts, 
    embeddings, 
    metadatas=metadatas, 
    persist_directory="enhanced_db"
)
enhanced_db.persist()

print("✅ Enhanced vector database created!")

In [ ]:
# Setup free language model
print("🧠 Setting up free language model...")

# Choose model based on available resources
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

try:
    # Try a good instruction-following model first
    model_name = "google/flan-t5-base"
    print(f"Loading {model_name}...")
    
    pipe = pipeline(
        "text2text-generation",
        model=model_name,
        device=0 if device == "cuda" else -1,
        max_new_tokens=200,
        temperature=0.1
    )
    
except Exception as e:
    print(f"Falling back to lighter model: {e}")
    # Fallback to smaller model
    model_name = "distilgpt2"
    pipe = pipeline(
        "text-generation",
        model=model_name,
        device=0 if device == "cuda" else -1,
        max_new_tokens=150,
        temperature=0.1,
        pad_token_id=50256
    )

llm = HuggingFacePipeline(pipeline=pipe)
print(f"✅ LLM ready: {model_name}")

In [ ]:
# Create enhanced RAG chain with better prompting
print("🔗 Creating enhanced RAG chain...")

# Enhanced prompt template
template = """You are Nur.AI, an expert assistant for Islamic heritage and civilization.

INSTRUCTIONS:
- Answer based ONLY on the provided context
- Focus on historical, cultural, and architectural facts
- Do NOT provide religious interpretations or rulings
- Be concise but informative
- If unsure, say "I don't have enough information"

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

# Create retriever with better settings
retriever = enhanced_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}  # Get top 4 relevant chunks
)

# Create QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

print("✅ Enhanced RAG chain ready!")

In [ ]:
# Enhanced query function with better formatting
def enhanced_query(question):
    """Enhanced query function with source formatting"""
    print(f"🔍 Searching: {question}")
    print("-" * 50)
    
    try:
        result = qa_chain({"query": question})
        
        # Display answer
        print("💡 ANSWER:")
        print(result["result"])
        print()
        
        # Display sources
        if result["source_documents"]:
            print("📚 SOURCES:")
            for i, doc in enumerate(result["source_documents"], 1):
                title = doc.metadata.get("title", "Unknown")
                url = doc.metadata.get("url", "")
                excerpt = doc.page_content[:150] + "..."
                
                print(f"{i}. {title}")
                if url:
                    print(f"   🔗 {url}")
                print(f"   📝 {excerpt}")
                print()
        
        return result
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

print("🎯 Enhanced query function ready!")

In [ ]:
# Test with sample questions
test_questions = [
    "What was the House of Wisdom in Baghdad?",
    "Describe the key features of Islamic architecture",
    "Who was Ibn Khaldun and what were his contributions?",
    "Tell me about the Alhambra palace"
]

print("🧪 TESTING ENHANCED RAG SYSTEM")
print("=" * 60)

for question in test_questions:
    enhanced_query(question)
    print("=" * 60)

In [ ]:
# Interactive chat interface
def interactive_chat():
    """Interactive chat with the enhanced RAG system"""
    print("\n🌙 Welcome to Nur.AI - Enhanced Islamic Heritage Explorer")
    print("Ask questions about Islamic history, art, and architecture")
    print("Type 'quit' to exit, 'help' for sample questions")
    print("=" * 60)
    
    sample_questions = [
        "What innovations came from the Islamic Golden Age?",
        "Describe Moorish architecture in Spain",
        "What was the role of madrasas in Islamic education?",
        "Tell me about Islamic calligraphy",
        "How did Islamic gardens influence design?"
    ]
    
    while True:
        try:
            question = input("\n🤔 Your question: ").strip()
            
            if question.lower() in ['quit', 'exit', 'bye']:
                print("👋 Thank you for exploring Islamic heritage!")
                break
                
            elif question.lower() == 'help':
                print("\n💡 Sample questions:")
                for i, q in enumerate(sample_questions, 1):
                    print(f"{i}. {q}")
                continue
                
            elif not question:
                continue
                
            enhanced_query(question)
            
        except KeyboardInterrupt:
            print("\n👋 Goodbye!")
            break
        except Exception as e:
            print(f"❌ Error: {e}")

# Start interactive chat
interactive_chat()

## 🚀 Next Steps

Your enhanced RAG system is now ready! Here's what you can do next:

### 📈 Expand the Dataset
- Add more Wikipedia articles
- Include UNESCO heritage site descriptions
- Add museum collection data

### 🔧 Improve the System
- Try different embedding models
- Experiment with chunk sizes
- Add metadata filtering

### 🌐 Deploy
- Use the Streamlit app for web interface
- Deploy on Hugging Face Spaces
- Create a mobile-friendly version

### 🎯 Specialize
- Focus on specific periods (e.g., Abbasid era)
- Add image search capabilities
- Include Arabic language support